# Variational Autoencoder for Generation of Antimicrobial Peptides

Implementation based on: **Dean & Walper, ACS Omega 2020, 5, 20746-20754**  
DOI: 10.1021/acsomega.0c00442

Architecture follows **Bowman et al. (2015)** "Generating Sentences from a Continuous Space":
- LSTM encoder/decoder with 1024 hidden units
- 2D latent space for visualization and interpolation
- KL annealing and character dropout
- 5-fold cross-validation, Adam optimizer, 250 epochs

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import KFold
import matplotlib.pyplot as plt
import random
from typing import List, Tuple, Optional, Dict

## Hyperparameters

All values match the paper's experimental section (Section 5).

In [ ]:
# --- Vocabulary ---
# 20 standard amino acids + X (unknown in APD3) + <pad>, <end>, <unk> = 24 tokens
# Paper: "binary vectors with length equal to the size of the amino acid vocabulary" = 24
VOCAB = [
    '<pad>', '<end>', '<unk>',
    'A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L',
    'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'X', 'Y'
]

# --- Architecture ---
MAX_SEQ_LEN = 12       # Paper: peptides truncated to <=12 amino acids
VOCAB_SIZE = len(VOCAB) # 24 (matches paper's data matrix: 5942 x 24 x 12)
HIDDEN_DIM = 1024       # Paper: "number of neurons for the LSTM layers... both set to 1024"
LATENT_DIM = 2          # Paper: 2D latent space (V1, V2 in figures)

# --- Training ---
BATCH_SIZE = 128
LEARNING_RATE = 1e-3    # Adam default
NUM_EPOCHS = 250        # Paper: "Training was stopped after 250 epochs"
N_FOLDS = 5             # Paper: "Five-fold cross-validation"
PATIENCE = 10           # Paper: "loss values did not decrease for 10 iterations"

# --- Bowman et al. techniques ---
WORD_DROPOUT_RATE = 0.5 # Character dropout: replace decoder input tokens with <unk>
KL_ANNEAL_EPOCHS = 20   # Linear annealing of KL weight from 0 to 1

# --- Reproducibility ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device(
    'cuda' if torch.cuda.is_available() else
    'mps' if torch.backends.mps.is_available() else 'cpu'
)
print(f'Device: {DEVICE}')

## Vocabulary and Preprocessing

Paper Section 5.1: sequences are tokenized (each residue separated) and represented
by a **one-hot encoding** scheme with vocabulary size 24. Sequences are padded/truncated
to length 12. Scrambled peptides are created by randomly shuffling the amino acid order.

In [ ]:
char_to_idx = {c: i for i, c in enumerate(VOCAB)}
idx_to_char = {i: c for i, c in enumerate(VOCAB)}

PAD_IDX = char_to_idx['<pad>']  # 0
END_IDX = char_to_idx['<end>']  # 1
UNK_IDX = char_to_idx['<unk>']  # 2


def tokenize(seq: str) -> List[int]:
    """Convert peptide string to token index list, padded to MAX_SEQ_LEN."""
    tokens = [char_to_idx.get(aa.upper(), UNK_IDX) for aa in seq]
    if len(tokens) < MAX_SEQ_LEN:
        tokens.append(END_IDX)
    while len(tokens) < MAX_SEQ_LEN:
        tokens.append(PAD_IDX)
    return tokens[:MAX_SEQ_LEN]


def detokenize(tokens) -> str:
    """Convert token indices back to peptide string."""
    if isinstance(tokens, torch.Tensor):
        tokens = tokens.cpu().tolist()
    seq = []
    for idx in tokens:
        char = idx_to_char.get(idx, '')
        if char in ('<end>', '<pad>'):
            break
        if char == '<unk>':
            continue
        seq.append(char)
    return ''.join(seq)


def one_hot_encode(tokens: List[int]) -> np.ndarray:
    """One-hot encode token sequence -> (MAX_SEQ_LEN, VOCAB_SIZE)."""
    oh = np.zeros((MAX_SEQ_LEN, VOCAB_SIZE), dtype=np.float32)
    for i, idx in enumerate(tokens):
        oh[i, idx] = 1.0
    return oh


def scramble_sequence(seq: str) -> str:
    """Randomly shuffle amino acid order (preserves composition and length)."""
    chars = list(seq)
    random.shuffle(chars)
    return ''.join(chars)


def preprocess_sequences(sequences: List[str]) -> torch.Tensor:
    """Convert list of peptide strings to one-hot tensor (N, MAX_SEQ_LEN, VOCAB_SIZE)."""
    encoded = [one_hot_encode(tokenize(s)) for s in sequences]
    return torch.tensor(np.array(encoded), dtype=torch.float32)


# Quick test
test_seq = 'RKLKKLWRKFR'
tokens = tokenize(test_seq)
print(f'Sequence: {test_seq}')
print(f'Tokens:   {tokens}')
print(f'Decoded:  {detokenize(tokens)}')
print(f'One-hot shape: {one_hot_encode(tokens).shape}')

## Dataset

Paper Section 5.1: 2971 AMP sequences from APD3, each paired with a randomly
scrambled copy, totaling 5942 training sequences. The scrambled peptides serve as
negative controls — same amino acid composition but destroyed sequence order.

In [ ]:
class AMPDataset(Dataset):
    """Dataset for antimicrobial peptide sequences (active + scrambled pairs)."""

    def __init__(self, sequences: Optional[List[str]] = None,
                 include_scrambled: bool = True):
        """
        Args:
            sequences: List of active peptide strings. If None, uses demo data.
            include_scrambled: If True, adds scrambled pair for each sequence.
        """
        if sequences is None:
            # Demo sequences from the paper's Table 1
            sequences = [
                'RKLKKLWRKFR', 'RRFVKKVRKLVK', 'FRWLRKWFRR',
                'VIREHKYVLLL', 'LPKIKKTVSTR', 'LLKSGRLLMKI',
                'KKIKRFLRKIG', 'GLGIIPHRRYGK', 'GIMSLFKGVLKT',
                'GLFKIIKNIFSG', 'KLFRIIKRIFKG'
            ]

        all_seqs = list(sequences)
        self.labels = [1] * len(all_seqs)  # 1 = active

        if include_scrambled:
            scrambled = [scramble_sequence(s) for s in sequences]
            all_seqs.extend(scrambled)
            self.labels.extend([0] * len(scrambled))  # 0 = scrambled

        self.sequences = all_seqs
        self.data = preprocess_sequences(all_seqs)
        self.labels = torch.tensor(self.labels, dtype=torch.long)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]


# Quick test with demo data
demo_dataset = AMPDataset()
print(f'Dataset size: {len(demo_dataset)}')
print(f'Data tensor shape: {demo_dataset.data.shape}')
print(f'First 3 active:    {demo_dataset.sequences[:3]}')
n_active = len(demo_dataset) // 2
print(f'First 3 scrambled: {demo_dataset.sequences[n_active:n_active+3]}')

## Model Architecture

Paper Section 5.2 + Bowman et al. (2015):

**Encoder**: LSTM (1024 hidden) reads one-hot sequence, final hidden state is projected
to latent distribution parameters $\mu$ and $\log\sigma^2$ (2D).

**Reparameterization**: $z = \mu + \sigma \cdot \epsilon$, where $\epsilon \sim \mathcal{N}(0, I)$

**Decoder**: Latent $z$ initializes LSTM hidden state (1024). Teacher-forced decoding
with **character dropout** — random input tokens are replaced with `<unk>` to force
reliance on the latent code rather than copying the input.

In [ ]:
class Encoder(nn.Module):
    """LSTM encoder: one-hot peptide -> latent distribution (mu, logvar)."""

    def __init__(self, vocab_size: int, hidden_dim: int, latent_dim: int):
        super().__init__()
        self.lstm = nn.LSTM(vocab_size, hidden_dim, batch_first=True)
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

    def forward(self, x: torch.Tensor):
        # x: (batch, seq_len, vocab_size)
        _, (h_n, _) = self.lstm(x)   # h_n: (1, batch, hidden_dim)
        h = h_n.squeeze(0)            # (batch, hidden_dim)
        return self.fc_mu(h), self.fc_logvar(h)


class Decoder(nn.Module):
    """LSTM decoder: latent z -> reconstructed peptide sequence."""

    def __init__(self, vocab_size: int, hidden_dim: int, latent_dim: int):
        super().__init__()
        self.vocab_size = vocab_size
        # Project z to LSTM initial hidden and cell states
        self.fc_h = nn.Linear(latent_dim, hidden_dim)
        self.fc_c = nn.Linear(latent_dim, hidden_dim)
        # Decoder LSTM and output projection
        self.lstm = nn.LSTM(vocab_size, hidden_dim, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, z: torch.Tensor, target_onehot: torch.Tensor,
                word_dropout_rate: float = 0.0):
        """
        Teacher-forced decoding with character dropout (Bowman et al.).

        Args:
            z: latent vector (batch, latent_dim)
            target_onehot: ground truth one-hot (batch, seq_len, vocab_size)
            word_dropout_rate: probability of replacing input token with <unk>
        """
        batch_size, seq_len, _ = target_onehot.shape

        # Initialize LSTM state from z
        h_0 = torch.tanh(self.fc_h(z)).unsqueeze(0)  # (1, batch, hidden)
        c_0 = torch.tanh(self.fc_c(z)).unsqueeze(0)

        # Shifted-right teacher forcing: input[t] = target[t-1], input[0] = zeros
        start = torch.zeros(batch_size, 1, self.vocab_size, device=z.device)
        decoder_input = torch.cat([start, target_onehot[:, :-1, :]], dim=1)

        # Character dropout (Bowman et al.): replace random positions with <unk>
        if self.training and word_dropout_rate > 0:
            mask = torch.rand(batch_size, seq_len, 1, device=z.device) < word_dropout_rate
            unk_oh = torch.zeros(1, 1, self.vocab_size, device=z.device)
            unk_oh[0, 0, UNK_IDX] = 1.0
            decoder_input = torch.where(mask, unk_oh.expand_as(decoder_input), decoder_input)

        output, _ = self.lstm(decoder_input, (h_0, c_0))
        return self.fc_out(output)  # (batch, seq_len, vocab_size)

    def generate(self, z: torch.Tensor, max_len: int = MAX_SEQ_LEN,
                 temperature: float = 1.0, stochastic: bool = False):
        """
        Autoregressive generation from latent z.

        Paper: "decoding from the same point in the latent space may result in
        a different peptide being generated" — use stochastic=True for this.
        """
        batch_size = z.shape[0]
        h = torch.tanh(self.fc_h(z)).unsqueeze(0)
        c = torch.tanh(self.fc_c(z)).unsqueeze(0)
        inp = torch.zeros(batch_size, 1, self.vocab_size, device=z.device)

        generated = []
        for _ in range(max_len):
            out, (h, c) = self.lstm(inp, (h, c))
            logits = self.fc_out(out)  # (batch, 1, vocab_size)

            if stochastic:
                probs = F.softmax(logits.squeeze(1) / temperature, dim=-1)
                token_idx = torch.multinomial(probs, 1)  # (batch, 1)
            else:
                token_idx = logits.argmax(dim=-1)  # (batch, 1)

            generated.append(token_idx)
            inp = torch.zeros(batch_size, 1, self.vocab_size, device=z.device)
            inp.scatter_(2, token_idx.unsqueeze(-1), 1.0)

        return torch.cat(generated, dim=1)  # (batch, max_len)

In [ ]:
class PeptideVAE(nn.Module):
    """
    VAE for antimicrobial peptide generation (Dean & Walper 2020).

    LSTM encoder -> 2D latent space -> LSTM decoder.
    Trained with KL annealing + character dropout (Bowman et al. 2015).
    """

    def __init__(self, vocab_size: int = VOCAB_SIZE,
                 hidden_dim: int = HIDDEN_DIM,
                 latent_dim: int = LATENT_DIM):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder = Encoder(vocab_size, hidden_dim, latent_dim)
        self.decoder = Decoder(vocab_size, hidden_dim, latent_dim)

    def reparameterize(self, mu: torch.Tensor, logvar: torch.Tensor):
        """Reparameterization trick: z = mu + sigma * epsilon."""
        if self.training:
            std = torch.exp(0.5 * logvar)
            return mu + std * torch.randn_like(std)
        return mu

    def forward(self, x: torch.Tensor, word_dropout_rate: float = 0.0):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        logits = self.decoder(z, x, word_dropout_rate)
        return logits, mu, logvar

    @torch.no_grad()
    def encode(self, x: torch.Tensor):
        """Encode to latent mean (for interpolation)."""
        self.eval()
        mu, _ = self.encoder(x)
        return mu

    @torch.no_grad()
    def decode(self, z: torch.Tensor, **kwargs):
        """Decode latent vector to token indices."""
        self.eval()
        return self.decoder.generate(z, **kwargs)

    @torch.no_grad()
    def interpolate(self, x1: torch.Tensor, x2: torch.Tensor,
                    steps: List[float] = [0.00, 0.33, 0.66, 0.99]):
        """
        Antimicrobial concept vector interpolation (paper Section 2.2).

        Linearly interpolates between two sequences in latent space:
        z = (1 - alpha) * z1 + alpha * z2
        """
        self.eval()
        mu1 = self.encode(x1)
        mu2 = self.encode(x2)

        results = []
        for alpha in steps:
            z = (1 - alpha) * mu1 + alpha * mu2
            tokens = self.decode(z)
            results.append((alpha, detokenize(tokens[0])))
        return results


# Verify model creation
model = PeptideVAE().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {n_params:,}')
print(f'Latent dimensions: {model.latent_dim}')

# Test forward pass
dummy = torch.randn(2, MAX_SEQ_LEN, VOCAB_SIZE).to(DEVICE)
logits, mu, logvar = model(dummy)
print(f'Input shape:  {dummy.shape}')
print(f'Output shape: {logits.shape}')
print(f'Latent mu:    {mu.shape}')
del model, dummy, logits, mu, logvar

## Loss Function

Paper Section 5.2: "The loss function was composed of **reconstruction loss** and **KL loss**
to penalize poor reconstruction of the data by the decoder and encoder output
representations of z that are different from a standard normal distribution."

$$\mathcal{L} = \mathcal{L}_{\text{recon}} + \beta \cdot D_{KL}\big(q(z|x) \,||\, p(z)\big)$$

where $\beta$ is linearly annealed from 0 to 1 (KL annealing, Bowman et al.).

In [ ]:
def vae_loss(logits, targets, mu, logvar, kl_weight=1.0):
    """
    VAE loss = Reconstruction (cross-entropy) + KL divergence.

    Args:
        logits: decoder output (batch, seq_len, vocab_size)
        targets: target one-hot (batch, seq_len, vocab_size)
        mu, logvar: latent distribution parameters
        kl_weight: annealing weight for KL term
    Returns:
        total_loss, recon_loss, kl_loss
    """
    # Reconstruction: cross-entropy (ignoring padding)
    target_idx = targets.argmax(dim=-1)  # (batch, seq_len)
    recon_loss = F.cross_entropy(
        logits.reshape(-1, logits.size(-1)),
        target_idx.reshape(-1),
        ignore_index=PAD_IDX,
        reduction='mean'
    )

    # KL divergence: D_KL(N(mu, sigma) || N(0,1))
    # = -0.5 * sum(1 + log(sigma^2) - mu^2 - sigma^2)
    kl_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

    total = recon_loss + kl_weight * kl_loss
    return total, recon_loss, kl_loss


def get_kl_weight(epoch: int, anneal_epochs: int = KL_ANNEAL_EPOCHS) -> float:
    """Linear KL annealing: weight goes from 0 to 1 over anneal_epochs."""
    return min(1.0, epoch / max(1, anneal_epochs))

## Training

Paper Section 5.2:
- **Optimizer**: Adam (stochastic gradient descent)
- **Cross-validation**: 5-fold, used to select architecture (512 vs 1024 hidden)
- **Early stopping**: training stopped when loss didn't decrease for 10 epochs
- **Epochs**: 250 (loss decreased < 0.1 per 10 epochs)

In [ ]:
def train_epoch(model, dataloader, optimizer, epoch):
    """Train for one epoch. Returns dict of average metrics."""
    model.train()
    total_loss, total_recon, total_kl = 0.0, 0.0, 0.0
    n_batches = 0
    kl_w = get_kl_weight(epoch)

    for batch_x, _ in dataloader:
        batch_x = batch_x.to(DEVICE)
        optimizer.zero_grad()

        logits, mu, logvar = model(batch_x, word_dropout_rate=WORD_DROPOUT_RATE)
        loss, recon, kl = vae_loss(logits, batch_x, mu, logvar, kl_w)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        total_loss += loss.item()
        total_recon += recon.item()
        total_kl += kl.item()
        n_batches += 1

    return {
        'loss': total_loss / n_batches,
        'recon': total_recon / n_batches,
        'kl': total_kl / n_batches,
        'kl_weight': kl_w
    }


@torch.no_grad()
def evaluate(model, dataloader):
    """Evaluate model on a dataset. Returns dict of average metrics."""
    model.eval()
    total_loss, total_recon, total_kl = 0.0, 0.0, 0.0
    n_batches = 0

    for batch_x, _ in dataloader:
        batch_x = batch_x.to(DEVICE)
        logits, mu, logvar = model(batch_x, word_dropout_rate=0.0)
        loss, recon, kl = vae_loss(logits, batch_x, mu, logvar, kl_weight=1.0)

        total_loss += loss.item()
        total_recon += recon.item()
        total_kl += kl.item()
        n_batches += 1

    return {
        'loss': total_loss / n_batches,
        'recon': total_recon / n_batches,
        'kl': total_kl / n_batches
    }

In [ ]:
def train_model(dataset, n_folds=N_FOLDS, num_epochs=NUM_EPOCHS,
                patience=PATIENCE, save_path='best_vae.pt'):
    """
    Train VAE with K-fold cross-validation.
    Returns best model and training history for all folds.
    """
    kfold = KFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    fold_histories = []
    best_overall_loss = float('inf')
    best_model_state = None

    for fold, (train_ids, val_ids) in enumerate(kfold.split(dataset)):
        print(f'\n{"="*60}')
        print(f'Fold {fold + 1}/{n_folds}')
        print(f'{"="*60}')

        train_loader = DataLoader(
            dataset, batch_size=BATCH_SIZE,
            sampler=torch.utils.data.SubsetRandomSampler(train_ids)
        )
        val_loader = DataLoader(
            dataset, batch_size=BATCH_SIZE,
            sampler=torch.utils.data.SubsetRandomSampler(val_ids)
        )

        model = PeptideVAE().to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

        history = {
            'train_loss': [], 'val_loss': [],
            'train_recon': [], 'train_kl': [],
            'val_recon': [], 'val_kl': []
        }
        best_val_loss = float('inf')
        epochs_no_improve = 0

        for epoch in range(num_epochs):
            train_m = train_epoch(model, train_loader, optimizer, epoch)
            val_m = evaluate(model, val_loader)

            history['train_loss'].append(train_m['loss'])
            history['val_loss'].append(val_m['loss'])
            history['train_recon'].append(train_m['recon'])
            history['train_kl'].append(train_m['kl'])
            history['val_recon'].append(val_m['recon'])
            history['val_kl'].append(val_m['kl'])

            if (epoch + 1) % 25 == 0 or epoch == 0:
                print(f'  Epoch {epoch+1:3d} | '
                      f'Train: {train_m["loss"]:.4f} '
                      f'(R:{train_m["recon"]:.4f} KL:{train_m["kl"]:.4f}) | '
                      f'Val: {val_m["loss"]:.4f} | '
                      f'beta={train_m["kl_weight"]:.2f}')

            # Early stopping (paper: loss didn't decrease for 10 epochs)
            if val_m['loss'] < best_val_loss - 0.01:
                best_val_loss = val_m['loss']
                epochs_no_improve = 0
                if best_val_loss < best_overall_loss:
                    best_overall_loss = best_val_loss
                    best_model_state = {
                        k: v.cpu().clone()
                        for k, v in model.state_dict().items()
                    }
            else:
                epochs_no_improve += 1

            if epochs_no_improve >= patience:
                print(f'  Early stopping at epoch {epoch+1}')
                break

        fold_histories.append(history)
        print(f'  Fold {fold+1} best val loss: {best_val_loss:.4f}')

    # Load best model across all folds
    final_model = PeptideVAE().to(DEVICE)
    if best_model_state is not None:
        final_model.load_state_dict(
            {k: v.to(DEVICE) for k, v in best_model_state.items()}
        )
        if save_path:
            torch.save(best_model_state, save_path)
            print(f'\nBest model saved to {save_path}')

    return final_model, fold_histories

## Generation and Interpolation Utilities

Paper Section 5.2: "Sequence sampling was performed using **linear interpolation**
between active peptides and their scrambled pairs... four points were generated at
equally spaced increments of 0.33: 0.00, 0.33, 0.66, and 0.99."

The line connecting scrambled (0.0) to active (1.0) in latent space is the
**antimicrobial concept vector**.

In [ ]:
@torch.no_grad()
def encode_sequence(model, seq: str) -> np.ndarray:
    """Encode a single peptide to its latent mean (2D point)."""
    model.eval()
    x = preprocess_sequences([seq]).to(DEVICE)
    mu = model.encode(x)
    return mu.cpu().numpy()[0]


@torch.no_grad()
def decode_latent(model, z: np.ndarray, **kwargs) -> str:
    """Decode a 2D latent point to a peptide string."""
    model.eval()
    z_t = torch.tensor(z, dtype=torch.float32).unsqueeze(0).to(DEVICE)
    tokens = model.decode(z_t, **kwargs)
    return detokenize(tokens[0])


def interpolate_peptides(
    model, seq_scrambled: str, seq_active: str,
    steps=(0.00, 0.33, 0.66, 0.99)
) -> List[Tuple[float, str]]:
    """
    Antimicrobial concept vector interpolation.

    Encodes scrambled (position 0) and active (position 1) peptides,
    then decodes intermediate points along the connecting line.
    """
    z_scr = encode_sequence(model, seq_scrambled)
    z_act = encode_sequence(model, seq_active)

    results = []
    for alpha in steps:
        z = (1 - alpha) * z_scr + alpha * z_act
        peptide = decode_latent(model, z)
        results.append((alpha, peptide))
    return results


def generate_full_interpolation_dataset(
    model, active_seqs: List[str], scrambled_seqs: List[str],
    steps=(0.00, 0.33, 0.66, 0.99)
) -> Dict[str, List[str]]:
    """
    Paper Section 2.2: generate interpolated dataset for all pairs.
    2971 pairs x 6 points = 17,838 total sequences.

    Returns dict: {'scrambled': [...], '0.00': [...], ..., '0.99': [...], 'active': [...]}
    """
    result = {f'{s:.2f}': [] for s in steps}
    result['scrambled'] = list(scrambled_seqs)
    result['active'] = list(active_seqs)

    for scr, act in zip(scrambled_seqs, active_seqs):
        for alpha, peptide in interpolate_peptides(model, scr, act, steps):
            result[f'{alpha:.2f}'].append(peptide)

    return result

## Visualization

Paper Figure 3: latent space maps with antimicrobial concept vectors.
Paper Figure S1: training loss curves.

In [ ]:
def plot_training_history(histories: List[Dict], save_path=None):
    """Plot training/validation loss curves across all folds."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    for fold, h in enumerate(histories):
        axes[0].plot(h['train_loss'], alpha=0.7, label=f'Fold {fold+1} train')
        axes[0].plot(h['val_loss'], alpha=0.7, ls='--', label=f'Fold {fold+1} val')
    axes[0].set(xlabel='Epoch', ylabel='Loss', title='Total Loss')
    axes[0].legend(fontsize=7)

    for fold, h in enumerate(histories):
        axes[1].plot(h['train_recon'], alpha=0.7, label=f'Fold {fold+1}')
    axes[1].set(xlabel='Epoch', ylabel='Loss', title='Reconstruction Loss')
    axes[1].legend(fontsize=7)

    for fold, h in enumerate(histories):
        axes[2].plot(h['train_kl'], alpha=0.7, label=f'Fold {fold+1}')
    axes[2].set(xlabel='Epoch', ylabel='Loss', title='KL Divergence')
    axes[2].legend(fontsize=7)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


@torch.no_grad()
def plot_latent_space(model, dataset, save_path=None):
    """
    Visualize 2D latent space (paper Figure 3).
    Active peptides in red, scrambled in green.
    """
    model.eval()
    loader = DataLoader(dataset, batch_size=256, shuffle=False)
    all_mu, all_labels = [], []

    for batch_x, batch_labels in loader:
        mu = model.encode(batch_x.to(DEVICE))
        all_mu.append(mu.cpu().numpy())
        all_labels.append(batch_labels.numpy())

    all_mu = np.concatenate(all_mu)
    all_labels = np.concatenate(all_labels)

    fig, ax = plt.subplots(figsize=(8, 6))
    scr_mask = all_labels == 0
    act_mask = all_labels == 1

    ax.scatter(all_mu[scr_mask, 0], all_mu[scr_mask, 1],
               c='green', alpha=0.4, s=15, label='Scrambled')
    ax.scatter(all_mu[act_mask, 0], all_mu[act_mask, 1],
               c='red', alpha=0.4, s=15, label='Active')

    ax.set(xlabel='V1', ylabel='V2', title='Latent Space Representation')
    ax.legend()
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


def plot_interpolation(model, seq_scrambled, seq_active, label=''):
    """Visualize interpolation path in latent space with decoded sequences."""
    steps = [0.00, 0.33, 0.66, 0.99]
    z_scr = encode_sequence(model, seq_scrambled)
    z_act = encode_sequence(model, seq_active)

    fig, ax = plt.subplots(figsize=(8, 4))

    # Draw concept vector
    ax.annotate('', xy=z_act, xytext=z_scr,
                arrowprops=dict(arrowstyle='->', color='black', lw=2))
    ax.scatter(*z_scr, c='green', s=100, zorder=5, marker='*')
    ax.scatter(*z_act, c='red', s=100, zorder=5, marker='*')
    ax.annotate(f'SP ({seq_scrambled})', z_scr, fontsize=8,
                textcoords='offset points', xytext=(5, 10))
    ax.annotate(f'P ({seq_active})', z_act, fontsize=8,
                textcoords='offset points', xytext=(5, 10))

    # Plot interpolation points
    for alpha in steps:
        z = (1 - alpha) * z_scr + alpha * z_act
        peptide = decode_latent(model, z)
        ax.scatter(*z, c='blue', s=40, zorder=5)
        ax.annotate(f'{alpha:.2f}\n{peptide}', z, fontsize=7,
                    textcoords='offset points', xytext=(5, -15))

    ax.set(xlabel='V1', ylabel='V2',
           title=f'Antimicrobial Concept Vector {label}')
    plt.tight_layout()
    plt.show()

## Run Training

**Note**: Currently uses demo data from Table 1 in the paper.
Replace `AMPDataset(sequences=None)` with actual APD3 data when available:
```python
with open('apd3_sequences.txt') as f:
    active_sequences = [line.strip()[:12] for line in f]
dataset = AMPDataset(sequences=active_sequences, include_scrambled=True)
```

In [ ]:
# Create dataset (demo data — replace with APD3 sequences)
dataset = AMPDataset(sequences=None, include_scrambled=True)
print(f'Dataset: {len(dataset)} sequences')
print(f'Tensor shape: {dataset.data.shape}  (N, seq_len={MAX_SEQ_LEN}, vocab={VOCAB_SIZE})')

In [ ]:
# Train with 5-fold cross-validation
model, histories = train_model(dataset)

In [ ]:
# Plot training curves (paper Figure S1)
plot_training_history(histories)

In [ ]:
# Visualize latent space (paper Figure 3)
plot_latent_space(model, dataset)

In [ ]:
# Interpolation example (paper Section 2.2)
# Using first active peptide and its scrambled pair
n_active = len(dataset) // 2
seq_active = dataset.sequences[0]
seq_scrambled = dataset.sequences[n_active]

print(f'Active:    {seq_active}')
print(f'Scrambled: {seq_scrambled}')
print(f'\nAntimicrobial concept vector interpolation:')
print(f'{"Step":>6}  {"Sequence":<15}')
print('-' * 25)

results = interpolate_peptides(model, seq_scrambled, seq_active)
for step, peptide in results:
    print(f'{step:6.2f}  {peptide:<15}')

In [ ]:
# Visualize interpolation path in latent space
plot_interpolation(model, seq_scrambled, seq_active, label='Series 1')